In [1]:
!pip install transformers

In [2]:
!pip install spacy_fastlang

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.7/227.7 kB 20.1 MB/s eta 0:00:00


In [3]:
import torch
from tqdm.notebook import tqdm

from transformers import BertTokenizer
from torch.utils.data import TensorDataset

from transformers import BertForSequenceClassification
import pandas as pd
import numpy as np

In [4]:
import random

seed_val = 17
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
path = '/content/drive/MyDrive/Memoria_Risk_Classification/Datasets/'

df_actual = pd.read_csv(path + 'Full_TOSDR_2023.csv')
df_actual['text'] = df_actual['text'].fillna(' ')

In [7]:
import spacy_fastlang
import spacy

nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("language_detector")

def detect_lang(text):
  doc = nlp(text)
  return doc._.language

df_actual['idioma'] = df_actual['text'].apply(lambda x: detect_lang(x))

df_actual = df_actual[df_actual['idioma'] == 'en']
print(df_actual['idioma'].value_counts())

df = df_actual

en    4390
Name: idioma, dtype: int64


In [8]:
label_map = {
    'success': 0,
    'secondary': 0,
    'warning' : 1,
    'danger': 1,
}

df['label'] = df['label'].map(label_map)
print(df.shape)
df.head(3)

(4390, 7)


,Unnamed: 0,service,privacy_text,case_link,label,text,idioma
0,0,1password,The service does not guarantee that software e...,https://edit.tosdr.org/points/6727,1,any errors or defects will be corrected\r\nTer...,en
1,1,1password,This service provides archives of their terms ...,https://edit.tosdr.org/points/6728,0,The Change Log section below is not a part of ...,en
2,2,1password,The services will notify users if personal dat...,https://edit.tosdr.org/points/13405,0,"<p>In an event of a breach, we recognize our r...",en


In [9]:
df['label'].value_counts()

0    2861
1    1529
Name: label, dtype: int64

In [10]:
print(df.head())
print(df.shape)

   Unnamed: 0    service                                       privacy_text  \
0           0  1password  The service does not guarantee that software e...   
1           1  1password  This service provides archives of their terms ...   
2           2  1password  The services will notify users if personal dat...   
3           3  1password  This service assumes no liability for any loss...   
4           4  1password  Users should revisit the terms periodically, a...   

                             case_link  label  \
0   https://edit.tosdr.org/points/6727      1   
1   https://edit.tosdr.org/points/6728      0   
2  https://edit.tosdr.org/points/13405      0   
3   https://edit.tosdr.org/points/6724      0   
4   https://edit.tosdr.org/points/6723      0   

                                                text idioma  
0  any errors or defects will be corrected\r\nTer...     en  
1  The Change Log section below is not a part of ...     en  
2  <p>In an event of a breach, we recognize 

In [11]:
df['label'].value_counts()

0    2861
1    1529
Name: label, dtype: int64

In [12]:
import re

def clean_text(texto):
    texto = re.sub(r'<.*?>', ' ', texto)
    texto = re.sub(r'[\n\r\t]', ' ', texto)
    return texto

df['text'] = df['text'].map(clean_text)
df.head(3)

,Unnamed: 0,service,privacy_text,case_link,label,text,idioma
0,0,1password,The service does not guarantee that software e...,https://edit.tosdr.org/points/6727,1,any errors or defects will be corrected Terms...,en
1,1,1password,This service provides archives of their terms ...,https://edit.tosdr.org/points/6728,0,The Change Log section below is not a part of ...,en
2,2,1password,The services will notify users if personal dat...,https://edit.tosdr.org/points/13405,0,"In an event of a breach, we recognize our res...",en


In [13]:
df['clase'] = df['label']

In [14]:
df.head()

,Unnamed: 0,service,privacy_text,case_link,label,text,idioma,clase
0,0,1password,The service does not guarantee that software e...,https://edit.tosdr.org/points/6727,1,any errors or defects will be corrected Terms...,en,1
1,1,1password,This service provides archives of their terms ...,https://edit.tosdr.org/points/6728,0,The Change Log section below is not a part of ...,en,0
2,2,1password,The services will notify users if personal dat...,https://edit.tosdr.org/points/13405,0,"In an event of a breach, we recognize our res...",en,0
3,3,1password,This service assumes no liability for any loss...,https://edit.tosdr.org/points/6724,0,"AgileBits, Inc., its directors, employees, par...",en,0
4,4,1password,"Users should revisit the terms periodically, a...",https://edit.tosdr.org/points/6723,0,"At our discretion, we may make changes to this...",en,0


In [15]:
from sklearn.model_selection import train_test_split

X, X_test, y, y_test = train_test_split(df.index.values,
                                        df.clase.values,
                                        test_size=0.10,
                                        random_state=42,
                                        stratify=df.clase.values)

X_train, X_val, y_train, y_val = train_test_split(X,
                                                  y,
                                                  test_size=0.10,
                                                  random_state=42,
                                                  stratify=y)

In [16]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(3555,)
(396,)
(439,)


In [17]:
df['data_type'] = ['not_set']*df.shape[0]

df.loc[X_train, 'data_type'] = 'train'
df.loc[X_val, 'data_type'] = 'val'
df.loc[X_test, 'data_type'] = 'test'

In [18]:
df.groupby(['clase', 'data_type']).count()

Unnamed: 0  service  privacy_text  case_link  label  text  \
clase data_type                                                              
0     test              286      286           286        286    286   286   
      train            2317     2317          2317       2317   2317  2317   
      val               258      258           258        258    258   258   
1     test              153      153           153        153    153   153   
      train            1238     1238          1238       1238   1238  1238   
      val               138      138           138        138    138   138   

                 idioma  
clase data_type          
0     test          286  
      train        2317  
      val           258  
1     test          153  
      train        1238  
      val           138

In [19]:
from google.colab import files

df.to_csv('TOSDR_SPLIT_DATA.csv', index=False)
files.download('TOSDR_SPLIT_DATA.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>